# Week 01 — Python Solution Lab
## Units, Vectors & the Language of Motion

**Companion to `notebooks/Week_01.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_01.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P4` | Vector Magnitude and Direction | `arctan2`, quadrant safety |
| **L2 · Intermediate** | `P5` | Multi-Vector Addition | vectorised component sums, path plot |
| **L3 · Challenge** | `P9` | 3D Cross Product and Torque Preview | `np.cross`, 360° direction sweep |

---

## L1 · Basic — P4: Vector Magnitude and Direction

> **Problem (Week_01.ipynb, L1 — P4).** A displacement vector has components
> $d_x = -12.0$ m and $d_y = 5.0$ m. Find the magnitude of the displacement and the
> angle it makes with the positive $x$-axis (measured counterclockwise).

**Diagram → Principle.** The vector points left and up, so it lies in the **second quadrant**.

**Equation.** $|\vec d| = \sqrt{d_x^2 + d_y^2}$ and $\theta = \operatorname{atan2}(d_y, d_x)$.

**Hand prediction.** $\sqrt{144+25} = 13.0$ m; the reference angle is $\arctan(5/12) = 22.6^\circ$,
so $\theta = 180^\circ - 22.6^\circ = 157.4^\circ$.

**What Python adds.** `np.arctan` cannot know the quadrant — it returns $-22.6^\circ$, which is
$180^\circ$ wrong. `np.arctan2(dy, dx)` takes both signs and gets it right. This single habit
prevents a large fraction of sign errors for the rest of the semester.

In [ ]:
# ═══ W01 · L1 · P4 — Vector Magnitude and Direction ═══
import numpy as np

# --- MODEL --------------------------------------------------------------
d = np.array([-12.0, 5.0])          # (dx, dy) in metres

# --- PREDICT ------------------------------------------------------------
mag   = np.hypot(d[0], d[1])                    # sqrt(dx**2 + dy**2), overflow-safe
theta = np.degrees(np.arctan2(d[1], d[0])) % 360.0   # quadrant-aware

print(f"|d|   = {mag:.3f} m")
print(f"theta = {theta:.2f} deg  (CCW from +x axis)")

# --- VERIFY 1: why arctan2 and not arctan -------------------------------
naive = np.degrees(np.arctan(d[1] / d[0]))
print(f"\nnp.arctan  -> {naive:7.2f} deg   WRONG: reports a quadrant-4 direction")
print(f"np.arctan2 -> {theta:7.2f} deg   correct: quadrant 2")
print(f"difference -> {theta - naive:7.2f} deg   (exactly 180 deg, the classic sign trap)")

# --- VERIFY 2: round-trip magnitude+angle back into components ----------
back = mag * np.array([np.cos(np.radians(theta)), np.sin(np.radians(theta))])
assert np.allclose(back, d), "round-trip failed"
print(f"\nRebuilt components {back.round(6)} == original {d}")

# --- CHECK against the printed answer -----------------------------------
assert abs(mag - 13.0) < 0.05, "magnitude mismatch"
assert abs(theta - 157.4) < 0.1, "angle mismatch"
print("\n[OK] Matches textbook answer: 13.0 m at 157.4 deg")

## L2 · Intermediate — P5: Multi-Vector Addition

> **Problem (Week_01.ipynb, L2 — P5).** A robot arm moves through three successive
> displacements: $\vec d_1 = 0.40$ m at $30^\circ$, $\vec d_2 = 0.60$ m at $150^\circ$, and
> $\vec d_3 = 0.35$ m at $270^\circ$. Find the resultant displacement vector (magnitude and
> direction) using the component method.

**Diagram → Principle.** Displacements add tip-to-tail; the resultant is the vector from the
first tail to the last tip. Adding *magnitudes* is meaningless — only components add.

**Equation.** $\vec R = \sum_i m_i(\cos\theta_i,\ \sin\theta_i)$.

**Hand prediction.** $R_x = -0.174$ m, $R_y = 0.150$ m $\Rightarrow |R| = 0.230$ m at $139.3^\circ$.

**What Python adds.** Written as one matrix of magnitudes times a matrix of unit vectors, the
"component method" becomes a single line that scales to *any* number of displacements. The
cumulative sum then gives the arm's whole path for free, which we plot.

In [ ]:
# ═══ W01 · L2 · P5 — Multi-Vector Addition (component method, vectorised) ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL: magnitudes and directions as arrays -------------------------
mags   = np.array([0.40, 0.60, 0.35])          # metres
angles = np.array([30.0, 150.0, 270.0])        # degrees CCW from +x

# --- PREDICT: components in ONE line ------------------------------------
th   = np.radians(angles)
vecs = mags[:, None] * np.column_stack([np.cos(th), np.sin(th)])   # shape (3, 2)
R    = vecs.sum(axis=0)

print("Individual components (m):")
for i, (v, m, a) in enumerate(zip(vecs, mags, angles), start=1):
    print(f"  d{i}: {m:.2f} m @ {a:5.1f} deg  ->  ({v[0]:+.4f}, {v[1]:+.4f})")

Rmag = np.hypot(*R)
Rdir = np.degrees(np.arctan2(R[1], R[0])) % 360.0
print(f"\nResultant  R = ({R[0]:+.4f}, {R[1]:+.4f}) m")
print(f"           |R| = {Rmag:.4f} m at {Rdir:.2f} deg")

# --- VERIFY: the naive 'add the magnitudes' answer is badly wrong -------
print(f"\nSum of magnitudes = {mags.sum():.3f} m, but |R| = {Rmag:.3f} m")
print(f"  -> the arm ends up {mags.sum()/Rmag:.1f}x closer than the path length. "
      "Direction matters.")

# --- Path plot: cumulative sum gives the tip-to-tail walk ---------------
path = np.vstack([[0, 0], np.cumsum(vecs, axis=0)])
fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.plot(path[:, 0], path[:, 1], "o-", color="#1565c0", lw=2, label="tip-to-tail path")
ax.annotate("", xy=R, xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="#e65100", lw=2.5))
ax.text(R[0] / 2 - 0.06, R[1] / 2, "R", color="#e65100", fontsize=13, fontweight="bold")
ax.set_aspect("equal"); ax.grid(alpha=.3); ax.axhline(0, c="k", lw=.6); ax.axvline(0, c="k", lw=.6)
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("W01 P5 — three displacements and their resultant"); ax.legend()
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert np.allclose(R, [-0.1732, 0.1500], atol=2e-3), "resultant mismatch"
assert abs(Rmag - 0.229) < 2e-3 and abs(Rdir - 139.1) < 0.4
print("[OK] Matches textbook answer: 0.230 m at 139.3 deg")

## L3 · Challenge — P9: 3D Cross Product and Torque Preview

> **Problem (Week_01.ipynb, L3 — P9).** A wrench handle lies along $\vec r = (0.25, 0, 0)$ m.
> You push with $\vec F = (0, 0, -80)$ N. (a) Compute $\vec\tau = \vec r \times \vec F$.
> (b) Find $|\vec\tau|$. (c) With the same $80$ N magnitude, which push direction maximises
> the torque?

**Diagram → Principle.** Only the component of $\vec F$ **perpendicular** to $\vec r$ twists the
bolt. That is exactly what the cross product extracts.

**Equation.** $\vec\tau = \vec r\times\vec F$, $|\vec\tau| = rF\sin\phi$.

**Hand prediction.** $\vec\tau = (0, 20, 0)$ N·m, $|\vec\tau| = 20$ N·m; maximum whenever
$\vec F \perp \vec r$.

**What Python adds.** Part (c) asks an *optimisation* question. Instead of arguing it verbally,
we sweep all 360° of push direction in the plane perpendicular-capable directions and **plot**
$|\vec\tau|(\phi)$. The $\sin\phi$ curve, its maximum at $90^\circ$, and the zero at $\phi=0$
all appear at once — the geometry becomes visible rather than asserted.

In [ ]:
# ═══ W01 · L3 · P9 — Cross product, torque, and a direction sweep ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
r = np.array([0.25, 0.0,   0.0])     # m, bolt -> grip point
F = np.array([0.00, 0.0, -80.0])     # N, straight down

# --- (a) PREDICT: the cross product -------------------------------------
tau = np.cross(r, F)
print(f"(a) tau = r x F = {tau} N*m")

# --- (b) magnitude, two independent ways --------------------------------
tau_mag  = np.linalg.norm(tau)
phi      = np.arccos(np.dot(r, F) / (np.linalg.norm(r) * np.linalg.norm(F)))
tau_geom = np.linalg.norm(r) * np.linalg.norm(F) * np.sin(phi)
print(f"(b) |tau| = {tau_mag:.2f} N*m   (from components)")
print(f"    |tau| = {tau_geom:.2f} N*m   (from r*F*sin(phi), phi = {np.degrees(phi):.1f} deg)")
assert np.isclose(tau_mag, tau_geom), "the two routes must agree"

# --- (c) OPTIMISE: sweep the push direction in the x-z plane ------------
Fmag  = 80.0
angs  = np.linspace(0, 360, 721)                       # angle of F measured from +x
Fs    = Fmag * np.column_stack([np.cos(np.radians(angs)),
                                np.zeros_like(angs),
                                np.sin(np.radians(angs))])
taus  = np.linalg.norm(np.cross(r, Fs), axis=1)

best = angs[np.argmax(taus)]
print(f"\n(c) sweep over 80 N push directions in the xz-plane:")
print(f"    max |tau| = {taus.max():.2f} N*m at {best:.0f} deg from +x  (i.e. perpendicular to r)")
print(f"    min |tau| = {taus.min():.2f} N*m at 0/180 deg (pushing ALONG the handle does nothing)")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(angs, taus, color="#2e7d32", lw=2)
ax.axhline(tau_mag, ls="--", c="#e65100", label=f"the given push: {tau_mag:.0f} N*m")
ax.axvline(270, ls=":", c="grey")
ax.set_xlabel("direction of the 80 N push (deg from +x)")
ax.set_ylabel("|tau| (N*m)")
ax.set_title("W01 P9 — torque vs push direction:  |tau| = r F sin(phi)")
ax.set_xticks(np.arange(0, 361, 45)); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert np.allclose(tau, [0.0, 20.0, 0.0]), "torque vector mismatch"
assert abs(taus.max() - 20.0) < 1e-6, "the given push is already optimal"
print("\n[OK] Matches textbook answer: tau = (0, 20, 0) N*m, |tau| = 20 N*m,")
print("     and any F perpendicular to r attains the same 20 N*m maximum.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_01.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
